In [0]:
%run ./_utils

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

#### Create `openalex.publishers.openalex_publishers_snapshot` in same format as API

In [ ]:
df_transformed = (
    spark.read.table("openalex.publishers.publishers_api")
    # Convert BIGINT id to URL format for snapshot
    .withColumn("id", F.concat(F.lit("https://openalex.org/P"), F.col("id")))
    # Coalesce null arrays to empty arrays
    .withColumn("lineage", F.coalesce(F.col("lineage"), F.array()))
    .withColumn("alternate_titles", F.coalesce(F.col("alternate_titles"), F.array()))
    .withColumn("country_codes", F.coalesce(F.col("country_codes"), F.array()))
    .withColumn("roles", F.coalesce(F.col("roles"), F.array()))
    .withColumn("counts_by_year", F.coalesce(F.col("counts_by_year"), F.array()))
)

df_transformed.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("openalex.publishers.openalex_publishers_snapshot")

#### Export to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/publishers/`.

In [0]:
date_str = get_snapshot_date()
df = spark.read.table("openalex.publishers.openalex_publishers_snapshot")
export_partitioned_all_formats(spark, dbutils, df, date_str, "publishers", salt=False)